# HW 4 
## Feature Engineering and Ensemble
### Dirks Wright

goal: get above 0.96

In [1]:
### load packages
import os
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CatBoostClassifier = None
    CATBOOST_AVAILABLE = False
 

RANDOM_STATE = 222
N_JOBS = -1
VALID_SIZE = 0.20
TUNE_ROWS = 120_000
CV_FOLDS = 3
CV_ROWS = 40_000
PERMUTATION_ROWS = 4_000

### Load Data

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

id_col = 'id'
target_col = 'Irrigation_Need'

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
display(train_df.head())
display(train_df[target_col].value_counts().to_frame('count'))
display(train_df[target_col].value_counts(normalize=True).to_frame('proportion'))

Train shape: (630000, 21)
Test shape: (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


,count
Irrigation_Need,
Low,369917
Medium,239074
High,21009


,proportion
Irrigation_Need,
Low,0.587170
Medium,0.379483
High,0.033348


### Feature Engineering

In [3]:
### feature engineering
def add_features(df):
    df = df.copy()
    eps = 1e-6

    ### water availability measures
    df['Moisture_Deficit'] = 100 - df['Soil_Moisture']
    df['Effective_Water_mm'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['Water_per_Hectare'] = df['Effective_Water_mm'] / (df['Field_Area_hectare'] + eps)
    df['Previous_Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + eps)
    df['Rainfall_per_Hectare'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + eps)
    df['Rainfall_per_Sunlight'] = df['Rainfall_mm'] / (df['Sunlight_Hours'] + eps)

    ### dryness/evaporation-style interactions
    df['Heat_Dryness_Index'] = df['Temperature_C'] * (100 - df['Humidity'])
    df['Wind_Heat_Index'] = df['Wind_Speed_kmh'] * df['Temperature_C']
    df['Evaporation_Pressure'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + eps)
    df['Sunlight_Temperature'] = df['Sunlight_Hours'] * df['Temperature_C']

    ### soil condition interactions
    df['Soil_pH_Distance_From_Neutral'] = (df['Soil_pH'] - 7).abs()
    df['Salinity_Moisture'] = df['Electrical_Conductivity'] * df['Soil_Moisture']
    df['Organic_Moisture'] = df['Organic_Carbon'] * df['Soil_Moisture']
    df['Carbon_pH_Interaction'] = df['Organic_Carbon'] * df['Soil_pH']

    ### categorical grouping features
    df['Crop_Stage'] = df['Crop_Type'].astype(str) + '_' + df['Crop_Growth_Stage'].astype(str)
    df['Soil_Crop'] = df['Soil_Type'].astype(str) + '_' + df['Crop_Type'].astype(str)
    df['Irrigation_Source'] = df['Irrigation_Type'].astype(str) + '_' + df['Water_Source'].astype(str)
    df['Season_Region'] = df['Season'].astype(str) + '_' + df['Region'].astype(str)
    df['Mulch_Irrigation'] = df['Mulching_Used'].astype(str) + '_' + df['Irrigation_Type'].astype(str)

    ### non-linear cut points without assuming linear relationship.
    df['Moisture_Level'] = pd.cut(
        df['Soil_Moisture'],
        bins=[-np.inf, 20, 40, 60, 80, np.inf],
        labels=['very_low', 'low', 'medium', 'high', 'very_high']
    ).astype(str)
    df['Rainfall_Level'] = pd.cut(
        df['Rainfall_mm'],
        bins=[-np.inf, 500, 1000, 1500, 2000, np.inf],
        labels=['very_low', 'low', 'medium', 'high', 'very_high']
    ).astype(str)
    df['Temperature_Level'] = pd.cut(
        df['Temperature_C'],
        bins=[-np.inf, 15, 22, 29, 36, np.inf],
        labels=['cool', 'mild', 'warm', 'hot', 'very_hot']
    ).astype(str)

    return df

train_fe = add_features(train_df)
test_fe = add_features(test_df)

### print new features
new_features = [col for col in train_fe.columns if col not in train_df.columns]
print('Created', len(new_features), 'new features')
display(pd.DataFrame({'new_feature': new_features}))

Created 22 new features


,new_feature
0,Moisture_Deficit
1,Effective_Water_mm
2,Water_per_Hectare
3,Previous_Irrigation_per_Hectare
4,Rainfall_per_Hectare
5,Rainfall_per_Sunlight
6,Heat_Dryness_Index
7,Wind_Heat_Index
8,Evaporation_Pressure
9,Sunlight_Temperature


In [4]:
### preprocessing and train/validation split
baseline_features = [col for col in train_df.columns if col not in [id_col, target_col]]
feature_cols = [col for col in train_fe.columns if col not in [id_col, target_col]]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_fe[target_col])

train_idx, val_idx = train_test_split(
    train_fe.index,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = train_fe.loc[train_idx, feature_cols]
X_val = train_fe.loc[val_idx, feature_cols]
y_train = y[train_idx]
y_val = y[val_idx]
y_val_series = pd.Series(y_val, index=X_val.index)

baseline_X_train = train_df.loc[train_idx, baseline_features]
baseline_X_val = train_df.loc[val_idx, baseline_features]

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = [col for col in feature_cols if col not in categorical_cols]
baseline_categorical_cols = baseline_X_train.select_dtypes(include=['object', 'category']).columns.tolist()
baseline_numeric_cols = [col for col in baseline_features if col not in baseline_categorical_cols]

print('Class mapping:', dict(enumerate(label_encoder.classes_)))
print('Training rows:', len(X_train), '| Validation rows:', len(X_val))
print('Numeric features:', len(numeric_cols), '| Categorical features:', len(categorical_cols))

Class mapping: {0: 'High', 1: 'Low', 2: 'Medium'}
Training rows: 504000 | Validation rows: 126000
Numeric features: 25 | Categorical features: 16


In [5]:
### one hot encoding preprocessor
def make_ohe_preprocessor(cat_cols, num_cols, scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler(with_mean=False)))

    return ColumnTransformer(
        transformers=[
            ('num', Pipeline(numeric_steps), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_cols)
        ],
        remainder='drop'
    )


### ordinal encoding preprocessor
def make_ordinal_preprocessor(cat_cols, num_cols):
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), num_cols),
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
        ],
        remainder='drop',
        sparse_threshold=0,
        verbose_feature_names_out=False
    )


### tune sampling utility function
def make_tune_sample(X_data, y_data, max_rows=TUNE_ROWS):
    if max_rows is None or len(X_data) <= max_rows:
        return X_data, y_data

    _, X_sample, _, y_sample = train_test_split(
        X_data,
        y_data,
        test_size=max_rows,
        random_state=RANDOM_STATE,
        stratify=y_data
    )
    return X_sample, y_sample


### evaluation function
def score_predictions(y_true, preds):
    return {
        'accuracy': accuracy_score(y_true, preds),
        'balanced_accuracy': balanced_accuracy_score(y_true, preds),
        'weighted_f1': f1_score(y_true, preds, average='weighted'),
        'macro_f1': f1_score(y_true, preds, average='macro')
    }

### Test Feature Engineering


In [6]:
### feature set comparison
feature_test_rows = []

feature_test_sets = [
    ('Original features only', baseline_X_train, baseline_X_val, baseline_categorical_cols, baseline_numeric_cols),
    ('Engineered feature set', X_train, X_val, categorical_cols, numeric_cols)
]

for label, X_tr, X_va, cat_cols, num_cols in feature_test_sets:
    X_fit, y_fit = make_tune_sample(X_tr, y_train, TUNE_ROWS)
    model = Pipeline([
        ('preprocess', make_ordinal_preprocessor(cat_cols, num_cols)),
        ('model', HistGradientBoostingClassifier(
            max_iter=160,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=0.10,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ])
    model.fit(X_fit, y_fit)
    preds = model.predict(X_va)
    row = {'feature_set': label}
    row.update(score_predictions(y_val, preds))
    feature_test_rows.append(row)

feature_test_results = pd.DataFrame(feature_test_rows).sort_values('balanced_accuracy', ascending=False)
display(feature_test_results)

,feature_set,accuracy,balanced_accuracy,weighted_f1,macro_f1
0,Original features only,0.982595,0.965308,0.982593,0.96179
1,Engineered feature set,0.982190,0.964969,0.982203,0.95976


## Multiple Tuned Models

The candidate models differ in a meaningful way: a scaled linear model, an extremely randomized tree model, a histogram gradient boosting model with ordinal categorical encoding, and CatBoost with native categorical handling. Each family has at least two tuning choices. I compare those choices with stratified cross-validation using balanced accuracy as the main metric, then use the holdout validation split only as a final check on the CV-selected model and ensemble.

In [8]:
### build functions for models
### build logistic regression
def build_logistic(C=1.0):
    return Pipeline([
        ('preprocess', make_ohe_preprocessor(categorical_cols, numeric_cols, scale_numeric=True)),
        ('model', LogisticRegression(
            C=C,
            max_iter=700,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS
        ))
    ])


### build extra trees
def build_extra_trees(n_estimators=300, max_features='sqrt', min_samples_leaf=2):
    return Pipeline([
        ('preprocess', make_ohe_preprocessor(categorical_cols, numeric_cols, scale_numeric=False)),
        ('model', ExtraTreesClassifier(
            n_estimators=n_estimators,
            max_features=max_features,
            min_samples_leaf=min_samples_leaf,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS
        ))
    ])

### build histogram gradient boosting
def build_hist_gradient(max_iter=220, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.10):
    return Pipeline([
        ('preprocess', make_ordinal_preprocessor(categorical_cols, numeric_cols)),
        ('model', HistGradientBoostingClassifier(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes,
            l2_regularization=l2_regularization,
            class_weight='balanced',
            random_state=RANDOM_STATE
        ))
    ])

### build catboost
def build_catboost(iterations=350, learning_rate=0.05, depth=8, l2_leaf_reg=5):
    return CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        loss_function='MultiClass',
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        verbose=False,
        thread_count=N_JOBS
    )


candidate_specs = [
    {
        'name': 'Logistic balanced C=0.5',
        'family': 'LogisticRegression',
        'kind': 'sklearn',
        'builder': lambda: build_logistic(C=0.5),
        'notes': 'Scaled one-hot representation with stronger regularization.'
    },
    {
        'name': 'Logistic balanced C=2.0',
        'family': 'LogisticRegression',
        'kind': 'sklearn',
        'builder': lambda: build_logistic(C=2.0),
        'notes': 'Scaled one-hot representation with lighter regularization.'
    },
    {
        'name': 'ExtraTrees sqrt leaf=2',
        'family': 'ExtraTrees',
        'kind': 'sklearn',
        'builder': lambda: build_extra_trees(n_estimators=300, max_features='sqrt', min_samples_leaf=2),
        'notes': 'Randomized tree ensemble with conservative leaves.'
    },
    {
        'name': 'ExtraTrees 0.7 leaf=1',
        'family': 'ExtraTrees',
        'kind': 'sklearn',
        'builder': lambda: build_extra_trees(n_estimators=350, max_features=0.7, min_samples_leaf=1),
        'notes': 'More flexible randomized trees with more features per split.'
    },
    {
        'name': 'HistGB lr=0.06 leaves=31',
        'family': 'HistGradientBoosting',
        'kind': 'sklearn',
        'builder': lambda: build_hist_gradient(max_iter=220, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.10),
        'notes': 'Boosted trees with slower learning and balanced classes.'
    },
    {
        'name': 'HistGB lr=0.04 leaves=63',
        'family': 'HistGradientBoosting',
        'kind': 'sklearn',
        'builder': lambda: build_hist_gradient(max_iter=300, learning_rate=0.04, max_leaf_nodes=63, l2_regularization=0.05),
        'notes': 'Deeper boosted trees with lower learning rate.'
    }
]

if CATBOOST_AVAILABLE:
    candidate_specs.extend([
        {
            'name': 'CatBoost depth=6 lr=0.08',
            'family': 'CatBoost',
            'kind': 'catboost',
            'builder': lambda: build_catboost(iterations=300, learning_rate=0.08, depth=6, l2_leaf_reg=5),
            'notes': 'CatBoost with native categorical handling and moderate depth.'
        },
        {
            'name': 'CatBoost depth=8 lr=0.05',
            'family': 'CatBoost',
            'kind': 'catboost',
            'builder': lambda: build_catboost(iterations=400, learning_rate=0.05, depth=8, l2_leaf_reg=7),
            'notes': 'CatBoost with deeper trees and a lower learning rate.'
        }
    ])
else:
    print('CatBoost is not installed, so CatBoost candidates will be skipped.')

pd.DataFrame([{key: spec[key] for key in ['name', 'family', 'notes']} for spec in candidate_specs])

,name,family,notes
0,Logistic balanced C=0.5,LogisticRegression,Scaled one-hot representation with stronger re...
1,Logistic balanced C=2.0,LogisticRegression,Scaled one-hot representation with lighter reg...
2,ExtraTrees sqrt leaf=2,ExtraTrees,Randomized tree ensemble with conservative lea...
3,ExtraTrees 0.7 leaf=1,ExtraTrees,More flexible randomized trees with more featu...
4,HistGB lr=0.06 leaves=31,HistGradientBoosting,Boosted trees with slower learning and balance...
5,HistGB lr=0.04 leaves=63,HistGradientBoosting,Deeper boosted trees with lower learning rate.
6,CatBoost depth=6 lr=0.08,CatBoost,CatBoost with native categorical handling and ...
7,CatBoost depth=8 lr=0.05,CatBoost,CatBoost with deeper trees and a lower learnin...


In [9]:
### fit models
def fit_candidate(spec, X_fit, y_fit, X_eval=None, y_eval=None):
    model = spec['builder']()

    if spec['kind'] == 'catboost':
        fit_kwargs = {
            'cat_features': categorical_cols,
            'verbose': False
        }
        model.fit(X_fit, y_fit, **fit_kwargs)
    else:
        model.fit(X_fit, y_fit)

    return model


def predict_labels(model, X_data):
    return np.asarray(model.predict(X_data)).reshape(-1).astype(int)


### stratified cross-validation model comparison
X_cv, y_cv = make_tune_sample(X_train, y_train, CV_ROWS)
print('Rows used for cross-validation:', len(X_cv), '| Folds:', CV_FOLDS)

cv_rows = []
cv_splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for spec in candidate_specs:
    print('Cross-validating:', spec['name'])
    for fold, (cv_train_idx, cv_val_idx) in enumerate(cv_splitter.split(X_cv, y_cv), start=1):
        X_cv_train = X_cv.iloc[cv_train_idx]
        X_cv_val = X_cv.iloc[cv_val_idx]
        y_cv_train = y_cv[cv_train_idx]
        y_cv_val = y_cv[cv_val_idx]

        model = fit_candidate(spec, X_cv_train, y_cv_train, X_cv_val, y_cv_val)
        preds = predict_labels(model, X_cv_val)

        row = {
            'name': spec['name'],
            'family': spec['family'],
            'fold': fold,
            'notes': spec['notes']
        }
        row.update(score_predictions(y_cv_val, preds))
        cv_rows.append(row)

cv_results_df = pd.DataFrame(cv_rows)
cv_summary_df = (
    cv_results_df
    .groupby(['name', 'family', 'notes'], as_index=False)
    .agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        balanced_accuracy_mean=('balanced_accuracy', 'mean'),
        balanced_accuracy_std=('balanced_accuracy', 'std'),
        weighted_f1_mean=('weighted_f1', 'mean'),
        weighted_f1_std=('weighted_f1', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std')
    )
    .sort_values('balanced_accuracy_mean', ascending=False)
    .reset_index(drop=True)
)
display(cv_summary_df)

cv_lookup = cv_summary_df.set_index('name')


X_tune, y_tune = make_tune_sample(X_train, y_train, TUNE_ROWS)
print('Rows used for final training after CV tuning:', len(X_tune))

fitted_models = {}
candidate_probas = {}
candidate_rows = []

for spec in candidate_specs:
    print('Fitting:', spec['name'])
    model = fit_candidate(spec, X_tune, y_tune, X_val, y_val)
    preds = predict_labels(model, X_val)
    probas = model.predict_proba(X_val)

    row = {
        'name': spec['name'],
        'family': spec['family'],
        'notes': spec['notes'],
        'cv_balanced_accuracy_mean': cv_lookup.loc[spec['name'], 'balanced_accuracy_mean'],
        'cv_balanced_accuracy_std': cv_lookup.loc[spec['name'], 'balanced_accuracy_std'],
        'cv_weighted_f1_mean': cv_lookup.loc[spec['name'], 'weighted_f1_mean'],
        'cv_weighted_f1_std': cv_lookup.loc[spec['name'], 'weighted_f1_std'],
        'cv_macro_f1_mean': cv_lookup.loc[spec['name'], 'macro_f1_mean'],
        'cv_accuracy_mean': cv_lookup.loc[spec['name'], 'accuracy_mean']
    }
    row.update({f'holdout_{metric}': value for metric, value in score_predictions(y_val, preds).items()})
    candidate_rows.append(row)

    fitted_models[spec['name']] = model
    candidate_probas[spec['name']] = probas

results_df = pd.DataFrame(candidate_rows).sort_values('cv_balanced_accuracy_mean', ascending=False).reset_index(drop=True)
display(results_df)

Rows used for cross-validation: 40000 | Folds: 3
Cross-validating: Logistic balanced C=0.5
Cross-validating: Logistic balanced C=2.0
Cross-validating: ExtraTrees sqrt leaf=2
Cross-validating: ExtraTrees 0.7 leaf=1
Cross-validating: HistGB lr=0.06 leaves=31
Cross-validating: HistGB lr=0.04 leaves=63
Cross-validating: CatBoost depth=6 lr=0.08
Cross-validating: CatBoost depth=8 lr=0.05


,name,family,notes,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,weighted_f1_mean,weighted_f1_std,macro_f1_mean,macro_f1_std
0,HistGB lr=0.06 leaves=31,HistGradientBoosting,Boosted trees with slower learning and balance...,0.981075,0.000751,0.964661,0.004843,0.981101,0.000740,0.957349,0.002483
1,HistGB lr=0.04 leaves=63,HistGradientBoosting,Deeper boosted trees with lower learning rate.,0.980700,0.000189,0.962760,0.004461,0.980720,0.000164,0.956307,0.001575
2,CatBoost depth=8 lr=0.05,CatBoost,CatBoost with deeper trees and a lower learnin...,0.984200,0.000637,0.961849,0.001470,0.984139,0.000641,0.969796,0.000468
3,CatBoost depth=6 lr=0.08,CatBoost,CatBoost with native categorical handling and ...,0.984100,0.000761,0.961746,0.002184,0.984037,0.000766,0.969947,0.000792
4,ExtraTrees 0.7 leaf=1,ExtraTrees,More flexible randomized trees with more featu...,0.979025,0.001488,0.954082,0.001031,0.978976,0.001489,0.960139,0.000840
5,ExtraTrees sqrt leaf=2,ExtraTrees,Randomized tree ensemble with conservative lea...,0.953100,0.002305,0.924793,0.003101,0.953128,0.002299,0.928851,0.002427
6,Logistic balanced C=0.5,LogisticRegression,Scaled one-hot representation with stronger re...,0.889600,0.002254,0.901527,0.000979,0.892702,0.001852,0.821686,0.005240
7,Logistic balanced C=2.0,LogisticRegression,Scaled one-hot representation with lighter reg...,0.890200,0.002554,0.901245,0.000965,0.893214,0.002143,0.822722,0.005257


Rows used for final training after CV tuning: 120000
Fitting: Logistic balanced C=0.5
Fitting: Logistic balanced C=2.0
Fitting: ExtraTrees sqrt leaf=2
Fitting: ExtraTrees 0.7 leaf=1
Fitting: HistGB lr=0.06 leaves=31
Fitting: HistGB lr=0.04 leaves=63
Fitting: CatBoost depth=6 lr=0.08
Fitting: CatBoost depth=8 lr=0.05


,name,family,notes,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_weighted_f1_mean,cv_weighted_f1_std,cv_macro_f1_mean,cv_accuracy_mean,holdout_accuracy,holdout_balanced_accuracy,holdout_weighted_f1,holdout_macro_f1
0,HistGB lr=0.06 leaves=31,HistGradientBoosting,Boosted trees with slower learning and balance...,0.964661,0.004843,0.981101,0.000740,0.957349,0.981075,0.982262,0.964421,0.982267,0.960022
1,HistGB lr=0.04 leaves=63,HistGradientBoosting,Deeper boosted trees with lower learning rate.,0.962760,0.004461,0.980720,0.000164,0.956307,0.980700,0.982778,0.964281,0.982762,0.962567
2,CatBoost depth=8 lr=0.05,CatBoost,CatBoost with deeper trees and a lower learnin...,0.961849,0.001470,0.984139,0.000641,0.969796,0.984200,0.983952,0.958958,0.983881,0.968431
3,CatBoost depth=6 lr=0.08,CatBoost,CatBoost with native categorical handling and ...,0.961746,0.002184,0.984037,0.000766,0.969947,0.984100,0.983952,0.959636,0.983884,0.968693
4,ExtraTrees 0.7 leaf=1,ExtraTrees,More flexible randomized trees with more featu...,0.954082,0.001031,0.978976,0.001489,0.960139,0.979025,0.982968,0.958049,0.982904,0.966584
5,ExtraTrees sqrt leaf=2,ExtraTrees,Randomized tree ensemble with conservative lea...,0.924793,0.003101,0.953128,0.002299,0.928851,0.953100,0.961333,0.940099,0.961424,0.938020
6,Logistic balanced C=0.5,LogisticRegression,Scaled one-hot representation with stronger re...,0.901527,0.000979,0.892702,0.001852,0.821686,0.889600,0.886341,0.900174,0.889779,0.816541
7,Logistic balanced C=2.0,LogisticRegression,Scaled one-hot representation with lighter reg...,0.901245,0.000965,0.893214,0.002143,0.822722,0.890200,0.886516,0.900392,0.889922,0.817046


In [10]:
### analyze best model
best_model_name = results_df.iloc[0]['name']
best_model = fitted_models[best_model_name]
best_preds = predict_labels(best_model, X_val)

print('Best CV-tuned individual model:', best_model_name)
print('CV balanced accuracy mean:', round(results_df.iloc[0]['cv_balanced_accuracy_mean'], 5))
print('Holdout balanced accuracy:', round(results_df.iloc[0]['holdout_balanced_accuracy'], 5))
print('CV weighted F1 mean:', round(results_df.iloc[0]['cv_weighted_f1_mean'], 5))
print('Holdout weighted F1:', round(results_df.iloc[0]['holdout_weighted_f1'], 5))
print()
print(classification_report(y_val, best_preds, target_names=label_encoder.classes_))
display(pd.DataFrame(
    confusion_matrix(y_val, best_preds),
    index=[f'true_{name}' for name in label_encoder.classes_],
    columns=[f'pred_{name}' for name in label_encoder.classes_]
))

Best CV-tuned individual model: HistGB lr=0.06 leaves=31
CV balanced accuracy mean: 0.96466
Holdout balanced accuracy: 0.96442
CV weighted F1 mean: 0.9811
Holdout weighted F1: 0.98227

              precision    recall  f1-score   support

        High       0.90      0.93      0.91      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.96      0.96      0.96    126000
weighted avg       0.98      0.98      0.98    126000



,pred_High,pred_Low,pred_Medium
true_High,3911,0,291
true_Low,6,73554,423
true_Medium,442,1073,46300


### Evaluate Which Features Are Useful

In [11]:
### useful features
def display_model_importance(model, model_name):
    if hasattr(model, 'get_feature_importance'):
        importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': model.get_feature_importance()
        }).sort_values('importance', ascending=False)
        print('Model-based importance for', model_name)
        display(importance.head(20))
        return importance

    if hasattr(model, 'named_steps'):
        estimator = model.named_steps['model']
        preprocessor = model.named_steps['preprocess']
        feature_names = preprocessor.get_feature_names_out()

        if hasattr(estimator, 'feature_importances_'):
            importance = pd.DataFrame({
                'feature': feature_names,
                'importance': estimator.feature_importances_
            }).sort_values('importance', ascending=False)
            print('Model-based importance for', model_name)
            display(importance.head(20))
            return importance

        if hasattr(estimator, 'coef_'):
            coef_importance = np.abs(estimator.coef_).mean(axis=0)
            importance = pd.DataFrame({
                'feature': feature_names,
                'importance': coef_importance
            }).sort_values('importance', ascending=False)
            print('Coefficient-based importance for', model_name)
            display(importance.head(20))
            return importance

    print('This model does not expose built-in feature importance.')
    return None


model_importance = display_model_importance(best_model, best_model_name)

perm_X = X_val.sample(n=min(PERMUTATION_ROWS, len(X_val)), random_state=RANDOM_STATE)
perm_y = y_val_series.loc[perm_X.index]

perm = permutation_importance(
    best_model,
    perm_X,
    perm_y,
    scoring='balanced_accuracy',
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
)

perm_importance = pd.DataFrame({
    'feature': perm_X.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

print('Permutation importance on validation sample')
display(perm_importance.head(20))

This model does not expose built-in feature importance.
Permutation importance on validation sample


,feature,importance_mean,importance_std
11,Crop_Growth_Stage,0.282951,0.004797
19,Moisture_Deficit,0.205881,0.005098
16,Mulching_Used,0.189146,0.010669
5,Temperature_C,0.185075,0.001459
9,Wind_Speed_kmh,0.090537,0.004981
2,Soil_Moisture,0.066797,0.002320
7,Rainfall_mm,0.041796,0.001848
20,Effective_Water_mm,0.002536,0.000206
6,Humidity,0.000360,0.000305
0,Soil_Type,0.000216,0.000000


### Ensemble

In [12]:
### create ensemble from top families
family_order = ['CatBoost', 'HistGradientBoosting', 'ExtraTrees', 'LogisticRegression']
ensemble_members = []

for family in family_order:
    family_rows = results_df[results_df['family'] == family]
    if not family_rows.empty:
        ensemble_members.append(family_rows.iloc[0]['name'])

print('Ensemble members:')
for name in ensemble_members:
    print('-', name)

proba_stack = np.stack([candidate_probas[name] for name in ensemble_members])

equal_avg_proba = proba_stack.mean(axis=0)
equal_avg_preds = equal_avg_proba.argmax(axis=1)

member_scores = results_df.set_index('name').loc[ensemble_members, 'cv_balanced_accuracy_mean'].to_numpy()
weights = member_scores / member_scores.sum()
weighted_avg_proba = np.average(proba_stack, axis=0, weights=weights)
weighted_avg_preds = weighted_avg_proba.argmax(axis=1)

ensemble_rows = []
for label, preds in [
    ('Equal Probability Average', equal_avg_preds),
    ('Weighted Probability Average', weighted_avg_preds)
]:
    row = {'name': label, 'family': 'Ensemble', 'notes': 'Average class probabilities from CV-selected model families.'}
    row.update({f'holdout_{metric}': value for metric, value in score_predictions(y_val, preds).items()})
    ensemble_rows.append(row)

ensemble_results = pd.DataFrame(ensemble_rows)
comparison_results = pd.concat([
    results_df[results_df['name'].isin(ensemble_members)],
    ensemble_results
], ignore_index=True).sort_values('holdout_balanced_accuracy', ascending=False).reset_index(drop=True)

display(comparison_results)

best_individual_bal_acc = results_df.iloc[0]['holdout_balanced_accuracy']
best_ensemble_bal_acc = ensemble_results['holdout_balanced_accuracy'].max()
print('Best CV-tuned individual holdout balanced accuracy:', round(best_individual_bal_acc, 5))
print('Best CV-selected ensemble holdout balanced accuracy:', round(best_ensemble_bal_acc, 5))

if best_ensemble_bal_acc > best_individual_bal_acc:
    print('The CV-selected ensemble improved holdout validation performance over the best CV-tuned individual model.')
else:
    print('The CV-selected ensemble did not beat the best CV-tuned individual model on this holdout validation split, but it still gives a useful comparison for stacking/averaging.')

Ensemble members:
- CatBoost depth=8 lr=0.05
- HistGB lr=0.06 leaves=31
- ExtraTrees 0.7 leaf=1
- Logistic balanced C=0.5


,name,family,notes,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_weighted_f1_mean,cv_weighted_f1_std,cv_macro_f1_mean,cv_accuracy_mean,holdout_accuracy,holdout_balanced_accuracy,holdout_weighted_f1,holdout_macro_f1
0,HistGB lr=0.06 leaves=31,HistGradientBoosting,Boosted trees with slower learning and balance...,0.964661,0.004843,0.981101,0.000740,0.957349,0.981075,0.982262,0.964421,0.982267,0.960022
1,Equal Probability Average,Ensemble,Average class probabilities from CV-selected m...,NaN,NaN,NaN,NaN,NaN,NaN,0.983587,0.963691,0.983546,0.967024
2,Weighted Probability Average,Ensemble,Average class probabilities from CV-selected m...,NaN,NaN,NaN,NaN,NaN,NaN,0.983675,0.963611,0.983631,0.967288
3,CatBoost depth=8 lr=0.05,CatBoost,CatBoost with deeper trees and a lower learnin...,0.961849,0.001470,0.984139,0.000641,0.969796,0.984200,0.983952,0.958958,0.983881,0.968431
4,ExtraTrees 0.7 leaf=1,ExtraTrees,More flexible randomized trees with more featu...,0.954082,0.001031,0.978976,0.001489,0.960139,0.979025,0.982968,0.958049,0.982904,0.966584
5,Logistic balanced C=0.5,LogisticRegression,Scaled one-hot representation with stronger re...,0.901527,0.000979,0.892702,0.001852,0.821686,0.889600,0.886341,0.900174,0.889779,0.816541


Best CV-tuned individual holdout balanced accuracy: 0.96442
Best CV-selected ensemble holdout balanced accuracy: 0.96369
The CV-selected ensemble did not beat the best CV-tuned individual model on this holdout validation split, but it still gives a useful comparison for stacking/averaging.


In [15]:
### save test file for submission
CREATE_SUBMISSION = True
FINAL_TRAIN_ROWS = None 
ensemble_members = [name for name in ensemble_members if 'ExtraTrees' not in name]
SUBMISSION_FILE = 'hw4_feature_engineered_ensemble_submission.csv'

if CREATE_SUBMISSION:
    specs_by_name = {spec['name']: spec for spec in candidate_specs}
    X_final, y_final = make_tune_sample(train_fe[feature_cols], y, FINAL_TRAIN_ROWS)
    X_test_final = test_fe[feature_cols]

    final_test_probas = []
    for name in ensemble_members:
        print('Refitting for submission:', name)
        final_model = fit_candidate(specs_by_name[name], X_final, y_final)
        final_test_probas.append(final_model.predict_proba(X_test_final))

    final_stack = np.stack(final_test_probas)
    final_weights = results_df.set_index('name').loc[ensemble_members, 'cv_balanced_accuracy_mean'].to_numpy()
    final_weights = final_weights / final_weights.sum()
    final_avg_proba = np.average(final_stack, axis=0, weights=final_weights)
    final_preds = final_avg_proba.argmax(axis=1)
    final_labels = label_encoder.inverse_transform(final_preds)

    submission = pd.DataFrame({
        id_col: test_df[id_col],
        target_col: final_labels
    })
    submission.to_csv(SUBMISSION_FILE, index=False)
    print('Saved submission to', SUBMISSION_FILE)
    display(submission.head())

Refitting for submission: CatBoost depth=8 lr=0.05
Refitting for submission: HistGB lr=0.06 leaves=31
Refitting for submission: Logistic balanced C=0.5
Saved submission to hw4_feature_engineered_ensemble_submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
